# Распределённое обучение: карта трека

Этот конспект связывает ограничения памяти и времени с DDP, ZeRO и FSDP. Цель — научиться выбирать стратегию, понимать цену коммуникации и честно измерять шаг обучения, а не получить готовую реализацию внешней лабы.

После прохождения карты вы сможете оценить память состояния модели, объяснить жизненный цикл градиентов и параметров при разных стратегиях и перейти к judge-задачам 102–107.

## 1. Проблема: почему одной GPU не хватает

Память обучения состоит как минимум из четырёх частей:

- **parameters** — веса модели;
- **gradients** — градиенты весов;
- **optimizer states** — состояния оптимизатора, например моменты AdamW;
- **activations** — сохранённые результаты forward, нужные для backward.

Кроме них бывают буферы, временные тензоры коллективных операций и фрагментация аллокатора. Поэтому арифметика «байт на параметр» — полезная нижняя оценка состояния модели, но не оценка всей пиковой памяти.

Память меняется внутри шага: в forward быстро растут activations; в backward растут gradients, а сохранённые activations постепенно освобождаются; в optimizer step нужны все gradients. У AdamW состояния обычно материализуются после первого optimizer step, а caching allocator подготавливает блоки в первом шаге. Поэтому успешный первый шаг ещё не доказывает, что следующий не закончится OOM.

Для AdamW в чистом fp32 на каждый обучаемый параметр приходятся:

$$
\underbrace{4}_{\text{parameter}} +
\underbrace{4}_{\text{gradient}} +
\underbrace{4}_{\text{first moment }m} +
\underbrace{4}_{\text{second moment }v}
= 16\ \text{bytes/parameter}.
$$

AdamW не хранит отдельную копию веса только из-за decoupled weight decay; два fp32-момента — это и есть дополнительные состояния оптимизатора в этой модели подсчёта.

В модели расчёта этого трека принята схема bf16 mixed precision: параметры и градиенты имеют bf16, а обновление делается через fp32 master weights и fp32-моменты:

$$
\underbrace{2}_{\text{bf16 parameter}} +
\underbrace{2}_{\text{bf16 gradient}} +
\underbrace{4}_{\text{fp32 master weight}} +
\underbrace{4}_{\text{fp32 }m} +
\underbrace{4}_{\text{fp32 }v}
= 16\ \text{bytes/parameter}.
$$

Это базовая 16 B/parameter модель: 2 B bf16 parameters + 2 B bf16 gradients + 4 B fp32 master weights + 8 B fp32 moments. В распространённом варианте с накопителем градиента в fp32 одновременно нужен ещё один 4 B слот, то есть получается 20 B/parameter. Реальные фреймворки и optimizer/runtime могут хранить состояния иначе (например, bf16-оптимизатор), и тогда арифметика меняется; перед реальным расчётом нужно проверить реализацию.

Это согласуется с [Ultra-Scale Playbook: memory for weights/grads/optimizer states](https://huggingface.co/spaces/nanotron/ultrascale-playbook?section=memory_for_weights/grads/optimizer_states): bf16 ускоряет вычисления и обычно уменьшает activations, но само по себе не обязано уменьшать постоянную память model states относительно fp32. Отдельно проверяйте, есть ли fp32 gradient accumulation.

In [ ]:
GIB = 1024 ** 3

def state_size_gib(parameter_count, bytes_per_parameter):
    return parameter_count * bytes_per_parameter / GIB

models = {"85M": 85_000_000, "7B": 7_000_000_000}
layouts = {"AdamW fp32": 4 + 4 + 4 + 4, "AdamW bf16 mixed": 2 + 2 + 4 + 4 + 4}

for model_name, parameter_count in models.items():
    print(model_name)
    for layout_name, bytes_per_parameter in layouts.items():
        size = state_size_gib(parameter_count, bytes_per_parameter)
        print(f"  {layout_name}: {size:.2f} GiB ({bytes_per_parameter} B/parameter)")

Для 85M нижняя оценка состояния AdamW в 16 B/parameter модели — примерно 1.27 GiB, для 7B — примерно 104.31 GiB. С fp32 gradient accumulation это уже примерно 1.58 и 130.39 GiB соответственно.

Activations зависят не только от модели, но и от входа. Для простой mixed-precision Transformer-оценки из Playbook полезна формула $M_{act} \approx L \cdot S \cdot B \cdot h \cdot (34 + 5n_{heads}S/h)$ bytes, где $L$ — число слоёв, $S$ — sequence length, $B$ — batch size, $h$ — hidden size. Коэффициенты зависят от архитектуры и kernels, но направление надёжно: память activations линейна по batch size и примерно квадратична по sequence length.

Два независимых рычага: activation recomputation (gradient checkpointing) выбрасывает часть activations и пересчитывает их в backward, обменивая память на compute; gradient accumulation делит global batch на microbatches, снижая activation peak каждого microbatch. Accumulation не шардит model states и удерживает accumulated gradients между microbatches, поэтому итоговый peak нужно измерять, а не выводить только из размера microbatch.

## 2. DDP: полная реплика и синхронизация градиентов

DistributedDataParallel (DDP) запускает по процессу на rank и держит на каждом rank полную реплику **параметров, градиентов и состояний оптимизатора**. Каждый rank обрабатывает свой shard batch, а затем градиенты одинаковых параметров объединяются коллективной операцией all-reduce. После одинакового optimizer step реплики снова совпадают.

DDP шардит входные данные, но **не шардит состояние модели**. При фиксированном локальном batch рост world size увеличивает effective batch, а при фиксированном global batch не меняет его; throughput может вырасти, но при коммуникационном bottleneck также может снизиться, и ограничение памяти состояния одной реплики остаётся.

Градиенты группируются в buckets. Как только backward вычислил все градиенты очередного bucket, DDP может запустить его all-reduce, пока autograd продолжает считать более ранние слои. Такой overlap backward ↔ all-reduce скрывает часть коммуникации за вычислением. Если buckets слишком крупные, первый collective стартует поздно; если слишком мелкие, растут latency и накладные расходы на множество collective calls.

Упрощённый ASCII-таймлайн одного шага:

    время ───────────────────────────────────────────────▶
    Rank 0: forward | backward B | backward A | step
                               \____ all-reduce B ____/
                                            \__ all-reduce A __/
    Rank 1: forward | backward B | backward A | step

Backward следующего bucket перекрывается с сетью для уже готового bucket. Практический вход в тему: [официальная серия PyTorch по DDP](https://docs.pytorch.org/tutorials/intermediate/ddp_series_intro.html).

## 3. ZeRO stage 1/2/3: шардирование состояния

ZeRO устраняет повторение состояния между data-parallel ranks поэтапно:

1. **Stage 1** шардит optimizer states.
2. **Stage 2** дополнительно шардит gradients.
3. **Stage 3** дополнительно шардит parameters.

Stage 3 требует собирать параметры тогда, когда они нужны слою, поэтому экономия постоянной памяти обменивается на дополнительные коллективные операции и временный gathered working set. Если единица шардирования близка к одному Transformer block, за training step возникают all-gather для параметров и в forward, и в backward; prefetch может перекрыть часть этой коммуникации, но увеличивает временный peak.

Ниже $P$ — число параметров, $N$ — world size. Память дана для описанной выше bf16 mixed-precision раскладки: 2 B параметр, 2 B градиент и 12 B master weight + моменты. Activations, buffers и временные all-gather buffers не включены.

| Стратегия | Что шардится | Нижняя оценка памяти состояния на rank | Коммуникация на шаг |
|---|---|---:|---|
| DDP | ничего | $16P$ B | all-reduce всех градиентов |
| ZeRO-1 | optimizer states | $4P + 12P/N$ B | синхронизация градиентов + распространение обновлённых shards; порядок объёма как у DDP |
| ZeRO-2 | optimizer states + gradients | $2P + 14P/N$ B | reduce-scatter градиентов + all-gather обновлённых параметров; порядок объёма как у DDP |
| ZeRO-3 | optimizer states + gradients + parameters | $16P/N$ B | как ZeRO-2 + all-gather параметров по мере forward/backward |

Это модель постоянного состояния, а не гарантия peak memory. Если включён fp32 gradient accumulation, добавьте $4P$ к DDP и ZeRO-1, а к ZeRO-2 и ZeRO-3 — $4P/N$: накопитель относится к gradients и шардится начиная со stage 2. Реальный объём зависит от алгоритма collective, dtype передачи, prefetching, повторного сбора параметров и размера единицы оборачивания.

Activations не получают эту экономию: data-parallel ranks обрабатывают разные microbatches, поэтому их activations не являются дубликатами. Это ключ к диагностике OOM: ZeRO/FSDP устраняют pressure от model states, а длинный контекст и большой microbatch требуют recomputation, accumulation или другой оси parallelism.

Первоисточник: [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054). Для этого трека особенно полезны разделы с разбором потребления памяти model states и описанием трёх стадий ZeRO-DP; затем стоит прочитать раздел про residual memory, чтобы не принять формулы model states за полный peak.

## 4. FSDP: параметры собираются только на время работы

FullyShardedDataParallel (FSDP) выражает близкие идеи средствами PyTorch:

| FSDP sharding strategy | Поведение | Аналогия с ZeRO |
|---|---|---|
| SHARD_GRAD_OP | параметры остаются доступными после forward; gradients и optimizer states шардированы | аналогия с ZeRO stage 2 по составу шардируемого |
| FULL_SHARD | parameters, gradients и optimizer states шардированы | ZeRO stage 3 |

Для SHARD_GRAD_OP это аналогия по составу шардируемого (gradients + optimizer states), но жизненный цикл параметров различается: FSDP-unit держит параметры unsharded от forward до backward, поэтому пиковая память и расписание collectives не совпадают с классическим ZeRO-2.

Для FULL_SHARD перед вычислением обёрнутого модуля выполняется all-gather его параметров. Во время backward FSDP снова получает нужные параметры, вычисляет градиенты и делает reduce-scatter, оставляя каждому rank только его gradient shard. После освобождения полных параметров остаются shards.

auto_wrap_policy определяет единицу шардирования, а значит гранулярность all-gather/reduce-scatter и момент, когда full parameters можно освободить. Конкретное расписание также зависит от порядка выполнения, resharding и prefetching. Практическая гранулярность — крупный повторяемый блок модели, например Transformer block: это даёт достаточно большие сообщения и позволяет освобождать полные параметры блока после использования. Одна обёртка вокруг всей модели ухудшает peak и overlap; тысячи обёрток вокруг мелких слоёв увеличивают latency. Weight tying, embeddings и output head требуют отдельной проверки: нельзя механически разрезать связанные параметры.

Официальные материалы:

- [PyTorch FSDP tutorial](https://docs.pytorch.org/tutorials/intermediate/FSDP1_tutorial.html) — минимальный жизненный цикл обучения;
- [PyTorch FSDP API docs](https://docs.pytorch.org/docs/stable/fsdp.html) — стратегии, политики оборачивания и ограничения API.

## 5. Честные замеры шага

CUDA kernels и collective operations ставятся в очереди асинхронно относительно Python. Таймер вокруг Python-вызовов без torch.cuda.synchronize() часто измеряет только постановку работы в очередь. Для end-to-end ms/step нужна синхронизация **до начала** измеряемого интервала и **после завершения** шага.

В distributed-случае локальный synchronize измеряет только свой rank. Честный end-to-end ms/step на $N$ ranks требует `dist.barrier` перед стартом замера и времени по самому медленному rank; иначе rank 0 может показать оптимистичное время.

Первые шаги нужно исключить как warmup: инициализируются kernels, memory pools и коммуникационные соединения, а AdamW может впервые выделить optimizer states. Для устойчивого итогового числа обычно берут median по серии шагов: среднее сильнее смещается редкими выбросами от фоновой нагрузки или разовой инициализации. Вместе с median полезно сохранять распределение или percentiles, batch/sequence shape, world size, dtype и версию окружения.

Peak VRAM — отдельная метрика от ms/step: после warmup фиксируйте peak allocated/reserved memory на каждом rank и в отчёте берите максимум по ranks. Первый шаг полезно записать отдельно: OOM, появляющийся лишь после него, часто объясняется lazy allocation optimizer states, а не ростом activations.

In [ ]:
from statistics import mean, median

steady_step_ms = [101.2, 99.8, 100.5, 100.1, 180.0]
print(f"mean: {mean(steady_step_ms):.1f} ms")
print(f"median: {median(steady_step_ms):.1f} ms")

## 6. Куда это масштабируется

Data parallelism и шардирование model states — не вся карта. **Tensor parallelism** делит вычисление одного слоя между устройствами, **pipeline parallelism** распределяет последовательные группы слоёв и microbatches, **expert parallelism** распределяет экспертов MoE; для длинного контекста отдельны sequence/context parallelism. На больших масштабах эти оси комбинируют.

Практическая карта выбора: если OOM дают params/grads/optimizer states — сначала сравнивайте ZeRO/FSDP; если доминируют activations — сначала меняйте microbatch, recomputation и стратегию для sequence/context. Это ортогональные причины, поэтому FSDP сам по себе не исправляет activation OOM. Подробности и следующие оси — в [Hugging Face Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook).

## 7. Контрольные вопросы

Первые пять вопросов ниже скопированы из секции «Вопросы, на которые у тебя должны быть ответы после лабы» файла [modern_nlp_labs/distributed-train-lab/STUDY.md](https://github.com/vladislav-vasilenko/modern_nlp_labs/blob/main/distributed-train-lab/STUDY.md). Ответов в этом ноутбуке намеренно нет.

1. Почему DDP держит полную копию оптимизатора на каждом GPU и сколько байт на параметр стоит AdamW в fp32 / bf16 mixed?
2. Что шардят ZeRO-1/2/3 и какой ценой по коммуникации на шаг?
3. Чем FSDP FULL_SHARD отличается от SHARD_GRAD_OP и каким стейджам ZeRO они соответствуют?
4. Почему замер без torch.cuda.synchronize() даёт неверные ms/step?
5. Почему на 85M-модели и 2 GPU FULL_SHARD (скорее всего) медленнее DDP, и при каком масштабе баланс переворачивается?
6. Как bucketing позволяет перекрыть backward и all-reduce, и что меняется при слишком крупных или слишком мелких buckets?
7. Почему уровень FSDP auto wrapping влияет одновременно на peak memory, overlap и число collective operations?

## 8. Маршрут трека

Проходите материал в таком порядке:

1. **102 — Optimizer memory math:** закрепить арифметику памяти.
2. **103 — Ring all-reduce:** разобрать механику и объём коммуникации.
3. **104 — DDP gradient sync:** исследовать синхронизацию градиентов.
4. **105 — ZeRO shard states:** разложить состояния по ranks.
5. **106 — FSDP wrap policy:** выбрать осмысленную гранулярность оборачивания.
6. **107 — Honest step timing:** построить корректный протокол измерения.
7. **108 — DDP skeleton walkthrough:** пройти контракт внешней лабы без готовых решений TODO.
8. **109 — Results tooling and conclusions:** проверить формат результатов и рубрики выводов.
9. После этого — самостоятельно заполнить skeleton в modern_nlp_labs и только затем арендовать GPU. Эти действия выполняются вне torchcode-labs.